# Bước 9 – Hướng 2: dùng một ít nhãn ở mạng mới

Lý thuyết từng bước, kèm ví dụ số, nằm trong tài liệu "TADP-IDS – Lý thuyết bước 9" Claude đã gửi.

| Ô | Việc | Thời gian |
|---|---|---|
| 1 | Kiểm tra môi trường và mã nguồn | 1 phút |
| 2 | Huấn luyện 4 chuyên gia, rồi thử 3 cách dùng 100 / 500 / 1.000 / 5.000 nhãn trên 4 mạng mới | 15–30 phút |
| 3 | Tổng hợp | 1 phút |

**Không cần GPU (card đồ họa).** Kết quả lưu vào `MyDrive/TADP_Q1/results/b9_it_nhan/`. Nếu Colab ngắt ở Ô 2, chạy lại Ô 2: mạng nào đã xong sẽ được bỏ qua.

## Ô 1 – Kiểm tra môi trường và mã nguồn

In [1]:
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")
CODE_DIR = "/content/drive/MyDrive/TADP_Q1/code"
sys.path.insert(0, CODE_DIR)
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
try:
    import lightgbm, pyarrow
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "pyarrow"], check=True)
EXPECTED = {"__init__.py": 125, "cic_data.py": 8579, "config.py": 3263, "data.py": 9956, "experts.py": 5614,
            "few_label.py": 10950, "metrics.py": 4002, "preprocess.py": 3601, "shift.py": 3863}
ok = True
for f, size in EXPECTED.items():
    p = os.path.join(CODE_DIR, "tadp", f)
    real = os.path.getsize(p) if os.path.exists(p) else None
    if real != size:
        ok = False
        print(f"   SAI: {f} – mong đợi {size} byte, thực tế {real}")
print("Mã nguồn:", "ĐẦY ĐỦ, đúng phiên bản bước 9" if ok else "CÓ LỖI – báo lại cho Claude")

Mounted at /content/drive
Mã nguồn: ĐẦY ĐỦ, đúng phiên bản bước 9


## Ô 2 – Chạy thực nghiệm

In ra lần lượt:
- 4 dòng "chuyên gia …": mỗi mạng một chuyên gia LightGBM (cây tăng cường gradient), kèm ngưỡng nguồn;
- 4 dòng "[tên mạng] xong": mỗi mạng mới đã thử xong 3 cách (M1 chọn chuyên gia, M2 học mới, M3 kết hợp) ở 4 mức nhãn, mỗi mức bốc 3 lần.

In [2]:
import sys
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/TADP_Q1/code")
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
from tadp.config import get_config
from tadp.data import load_all, make_splits, common_features
from tadp.few_label import run_few_label
B = "/content/drive/MyDrive/TADP_Q1"
DOMAINS_CIC = {"CIC-IDS2017": ["2017"], "CSE-CIC-IDS2018": ["2018"], "CIC-ToN-IoT": ["ton"], "CIC-UNSW-NB15": ["unsw"]}
cfg = get_config(RAW_DIR=f"{B}/raw_cic", CACHE_DIR=f"{B}/cache_cic", RESULT_DIR=f"{B}/results/b9_it_nhan",
                 DOMAINS=DOMAINS_CIC, SPLIT_MODE="episode", LGBM_ROUNDS=200)
data = load_all(cfg)
splits, _ = make_splits(cfg, data)
cols = common_features(data, "no_id")
run_few_label(cfg, data, splits, cols, pv="P1_log", budgets=(100, 500, 1000, 5000), repeats=(0, 1, 2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CIC-IDS2017: 999,712 dòng, 87 cột, tỷ lệ tấn công = 0.2411
CSE-CIC-IDS2018: 999,829 dòng, 87 cột, tỷ lệ tấn công = 0.0559
CIC-ToN-IoT: 1,000,028 dòng, 82 cột, tỷ lệ tấn công = 0.5302
CIC-UNSW-NB15: 1,000,399 dòng, 82 cột, tỷ lệ tấn công = 0.0251
chuyên gia CIC-IDS2017     : ngưỡng nguồn = 0.394 (22 giây)
chuyên gia CSE-CIC-IDS2018 : ngưỡng nguồn = 0.939 (21 giây)
chuyên gia CIC-ToN-IoT     : ngưỡng nguồn = 0.524 (21 giây)
chuyên gia CIC-UNSW-NB15   : ngưỡng nguồn = 0.419 (17 giây)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

[CIC-IDS2017] xong trong 186 giây


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

[CSE-CIC-IDS2018] xong trong 190 giây


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

[CIC-ToN-IoT] xong trong 186 giây


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

[CIC-UNSW-NB15] xong trong 170 giây


,target,method,k,repeat,so_tan_cong_trong_k,nguong,macro_f1,auroc,fpr,macro_f1_constant
0,CIC-IDS2017,k0_trung_binh_deu,0,0,0,0.627179,0.439675,0.715125,0.005605,0.431467
1,CIC-IDS2017,k0_chon_theo_kiem_dinh_nguon,0,0,0,0.938566,0.638556,0.934467,0.000433,0.431467
2,CIC-IDS2017,k0_trong_so_TADP,0,0,0,0.882357,0.546811,0.788757,0.000063,0.431467
3,CIC-IDS2017,tham_khao_chuyen_gia_tot_nhat_biet_dap_an,0,0,0,0.002722,0.852720,0.934467,0.064412,0.431467
4,CIC-IDS2017,tran_du_nhan,200000,0,48335,0.393673,0.999576,0.999995,0.000032,0.431467
...,...,...,...,...,...,...,...,...,...,...
159,CIC-UNSW-NB15,M2_hoc_moi,5000,1,113,0.397543,0.850431,0.992240,0.012043,0.493640
160,CIC-UNSW-NB15,M3_ket_hop,5000,1,113,0.896189,0.714243,0.919111,0.016193,0.493640
161,CIC-UNSW-NB15,M1_chon_chuyen_gia,5000,2,146,0.980010,0.579997,0.628634,0.039133,0.493640
162,CIC-UNSW-NB15,M2_hoc_moi,5000,2,146,0.712341,0.836104,0.993605,0.008334,0.493640


## Ô 3 – Tổng hợp

- Bảng 9.1: Macro-F1 và AUROC trung bình 4 mạng mới, theo cách làm và số nhãn.
- Bảng 9.2: Macro-F1 theo từng mạng mới.
- Bảng 9.3: độ dao động giữa 3 lần bốc nhãn.

Chạy xong báo Claude.

In [3]:
import sys
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/TADP_Q1/code")
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
from tadp.config import get_config
from tadp.few_label import summarize_few_label
B = "/content/drive/MyDrive/TADP_Q1"
cfg = get_config(RAW_DIR=f"{B}/raw_cic", CACHE_DIR=f"{B}/cache_cic", RESULT_DIR=f"{B}/results/b9_it_nhan")
summarize_few_label(cfg)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BẢNG 9.1 – Trung bình 4 mạng đích (mỗi mạng: trung bình 3 lần bốc nhãn)
                                                   macro_f1  auroc  macro_f1_constant
method                                    k                                         
M1_chon_chuyen_gia                        100        0.625  0.688              0.439
                                          500        0.640  0.697              0.439
                                          1000       0.649  0.699              0.439
                                          5000       0.669  0.698              0.439
M2_hoc_moi                                100        0.657  0.805              0.439
                                          500        0.827  0.984              0.439
                                          1000       0.784  0.990              0.439
                                 

(                                                  macro_f1  auroc  macro_f1_constant
 method                                    k                                         
 M1_chon_chuyen_gia                        100        0.625  0.688              0.439
                                           500        0.640  0.697              0.439
                                           1000       0.649  0.699              0.439
                                           5000       0.669  0.698              0.439
 M2_hoc_moi                                100        0.657  0.805              0.439
                                           500        0.827  0.984              0.439
                                           1000       0.784  0.990              0.439
                                           5000       0.843  0.995              0.439
 M3_ket_hop                                100        0.650  0.765              0.439
                                           500        